In [42]:
import requests
import pandas as pd
import os
from datetime import datetime
import datetime
import numpy as np

In [26]:
# NOTES:
# 1Y of lookback = 252 trading days
# 5Y of lookback = 1260 trading days

In [4]:
ticker = 'AAPL'

In [39]:
def get_sd(ticker):

    # API Setup
    base_url = 'https://financialmodelingprep.com/api/v3/technical_indicator/1day/'
    api_key = 'ksnxCfuMd8YcYtNVDrqZKBY0aZeDMLX8'

    # Create 3 URLs for 1Y 5Y and 10Y
    url_252 = f'{base_url}{ticker}?type=standardDeviation&period=252&apikey={api_key}'
    url_1260 = f'{base_url}{ticker}?type=standardDeviation&period=1260&apikey={api_key}'
    url_2520 = f'{base_url}{ticker}?type=standardDeviation&period=2520&apikey={api_key}'

    # Get data from all 3 URLs
    response_252 = requests.get(url_252)
    response_1260 = requests.get(url_1260)
    response_2520 = requests.get(url_2520)
    
    data_252 = response_252.json()
    data_1260 = response_1260.json()
    data_2520 = response_2520.json()
    
    df_252 = pd.DataFrame(data_252)
    df_1260 = pd.DataFrame(data_1260)
    df_2520 = pd.DataFrame(data_2520)
    
    # Drop unnecessary columns
    df_252 = df_252.drop(axis = 1, columns =['high','low','volume', 'open','close'] )
    df_1260 = df_1260.drop(axis = 1, columns =['high','low','volume', 'open','close'] )
    df_2520 = df_2520.drop(axis = 1, columns =['high','low','volume', 'open','close'] )
    
    # Remove unnecessary rows
    df_252 = df_252.head(1)
    df_1260 = df_1260.head(1)
    df_2520 = df_2520.head(1)

    # Display data
    print("1Y Average SD:", df_252)
    print("5Y Average SD:", df_1260)
    print("10Y Average SD:", df_2520)

In [40]:
get_sd(ticker)

1Y Average SD:                   date  standardDeviation
0  2025-03-12 00:00:00          24.800494
5Y Average SD:                   date  standardDeviation
0  2025-03-12 00:00:00          41.544606
10Y Average SD:                   date  standardDeviation
0  2025-03-12 00:00:00          67.249676


In [43]:
def get_cagr(ticker, years):
    
    # API Setup
    base_url = 'https://financialmodelingprep.com/api/v3/historical-price-full/'
    api_key = 'ksnxCfuMd8YcYtNVDrqZKBY0aZeDMLX8'
    
    # Calculate date range
    end_date = datetime.date.today()
    start_date = end_date - datetime.timedelta(days=years*365)

    # FMP API Endpoint
    url = f"{base_url}{ticker}?from={start_date}&to={end_date}&apikey={api_key}"
    
    # Fetch data
    response = requests.get(url)
    data = response.json()

    if "historical" in data and len(data["historical"]) > 0:
        # Sort data by date (oldest to newest)
        historical_data = sorted(data["historical"], key=lambda x: x["date"])
        
        # Extract start and end prices
        P_start = historical_data[0]["close"]
        P_end = historical_data[-1]["close"]
        
        # Calculate CAGR
        CAGR = ((P_end / P_start) ** (1/years)) - 1
        return round(CAGR * 100, 2)  # Convert to percentage
    else:
        return "No data available"


cagr_1y = get_cagr(ticker, 1)
cagr_5y = get_cagr(ticker, 5)
cagr_10y = get_cagr(ticker, 10)

print(f"1Y CAGR: {cagr_1y}%")
print(f"5Y CAGR: {cagr_5y}%")
print(f"10Y CAGR: {cagr_10y}%")

1Y CAGR: 25.26%
5Y CAGR: 25.57%
10Y CAGR: 21.39%
